# PG-LIF — Phase P1 (v2): SHD Benchmark, corrected PG-LIF cell
**Project:** PG-LIF · **Phase:** P1 of the Research and Implementation Plan · **Feeds:** manuscript Table 1 (Section 7.1)

**Why v2.** The first P1 run produced chance-level PG-LIF accuracy (7.5%) while the baselines behaved normally. Diagnosis on a structured synthetic task (22.7% for the shipped cell vs. 100% after the fix) identified two implementation defects in `PGLIFCell`, not a property of the model: (i) feedforward input reached the soma **only** through the all-or-none plateau, an information bottleneck that contradicts the manuscript's theory, in which the plateau is an *additional* slow pathway; and (ii) the plateau drive carried a leftover physical-units factor (1−αm) ≈ 0.05 that weakened it ~20× relative to the convention used by every other cell. Both are fixed here: the soma now receives `I_ff + I_rec + κ·p`, and dendrite-only feedforward routing becomes ablation material. The v1 gate verdict is therefore void for PG-LIF; baseline results remain valid and are **copied automatically** from the v1 run folder so only PG-LIF retrains.

Trains PG-LIF and the baselines (LIF, ALIF, TC-LIF, DH-LIF) on **SHD** in an identical skeleton (700 → recurrent spiking layer → leaky readout, cross-entropy on summed readout potential). TC-LIF uses the official dynamics (github.com/ZhangShimin1/TC-LIF); DH-LIF is a re-implementation per Zheng et al. (2024).

**Requirements:** GPU runtime. SHD is reused from `My Drive/PG_LIF/data/SHD/`.

**Modes:** `PAPER_MODE = False` (default): T = 100 bins, 20 epochs, seed 0. `PAPER_MODE = True`: official TC-LIF protocol (T = 250, 100 epochs, LR drops at [40, 80], seeds 0–4) for Table 1. Finished (model, seed) pairs are skipped on rerun.

In [ ]:
# --- Setup: Drive, folders ---
import os, json, time, gzip, shutil, urllib.request
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/My Drive'
    if not os.path.isdir(ROOT):    # some environments expose the no-space alias only
        ROOT = '/content/drive/MyDrive'
    BASE = os.path.join(ROOT, 'PG_LIF')
except Exception:
    BASE = './PG_LIF'   # local fallback (development only)
DATA = os.path.join(BASE, 'data', 'SHD')
P1 = os.path.join(BASE, 'P1_results')
os.makedirs(DATA, exist_ok=True); os.makedirs(P1, exist_ok=True)
print('Base folder:', BASE)

In [ ]:
# --- CONFIG ---
PAPER_MODE = False        # True = official TC-LIF SHD protocol (T=250, 100 epochs, seeds 0-4)
MODELS = ['LIF', 'ALIF', 'TCLIF', 'DHLIF', 'PGLIF']   # subset to shorten a session
SEEDS  = [0, 1, 2, 3, 4] if PAPER_MODE else [0]
T_BINS   = 250 if PAPER_MODE else 100
EPOCHS   = 100 if PAPER_MODE else 20
SCHEDULE = [40, 80] if PAPER_MODE else [12]
BATCH  = 64
LR     = 5e-4
HIDDEN = 128
MAX_TIME = 1.4            # seconds of each SHD sample used
N_IN, N_OUT = 700, 20
RUN_TAG = ('paper' if PAPER_MODE else 'quick') + f'_T{T_BINS}_E{EPOCHS}_H{HIDDEN}_v2'
OUT = os.path.join(P1, RUN_TAG); os.makedirs(OUT, exist_ok=True)
# Reuse v1 baseline results (the PG-LIF fix does not affect them): copy json + checkpoint if present
PREV = os.path.join(P1, RUN_TAG[:-3])
if os.path.isdir(PREV):
    for m in MODELS:
        if m == 'PGLIF': continue
        for sd in SEEDS:
            for suf in [f'{m}_s{sd}.json', f'{m}_s{sd}_best.pt']:
                src, dst = os.path.join(PREV, suf), os.path.join(OUT, suf)
                if os.path.exists(src) and not os.path.exists(dst):
                    shutil.copy(src, dst); print('reused from v1:', suf)
with open(os.path.join(OUT, 'config.json'), 'w') as f:
    json.dump({k: v for k, v in globals().items() if k in
               ['PAPER_MODE','MODELS','SEEDS','T_BINS','EPOCHS','SCHEDULE','BATCH','LR','HIDDEN','MAX_TIME']},
              f, indent=2, default=str)
print('Run folder:', OUT)

## Data: download and cache SHD, dense binning
Official files from the Zenke lab (`zenkelab.org/datasets`). Events are binned into `T_BINS` steps over the first `MAX_TIME` seconds (the standard SpyTorch/TC-LIF preprocessing). Dense batches are built on the fly to keep RAM modest.

In [ ]:
import numpy as np, h5py
URLS = {'shd_train.h5': 'https://zenkelab.org/datasets/shd_train.h5.gz',
        'shd_test.h5':  'https://zenkelab.org/datasets/shd_test.h5.gz'}
for name, url in URLS.items():
    dst = os.path.join(DATA, name)
    if not os.path.exists(dst):
        gz = dst + '.gz'
        print('downloading', url)
        urllib.request.urlretrieve(url, gz)
        with gzip.open(gz, 'rb') as fi, open(dst, 'wb') as fo: shutil.copyfileobj(fi, fo)
        os.remove(gz)
    print(name, 'ready,', os.path.getsize(dst)//(1<<20), 'MB')

def load_split(fname):
    with h5py.File(os.path.join(DATA, fname), 'r') as f:
        times = [np.array(t) for t in f['spikes']['times']]
        units = [np.array(u) for u in f['spikes']['units']]
        labels = np.array(f['labels'], dtype=np.int64)
    return times, units, labels

TR = load_split('shd_train.h5'); TE = load_split('shd_test.h5')
print('train samples:', len(TR[2]), '| test samples:', len(TE[2]))

def batches(split, batch_size, shuffle, T=None, device='cpu'):
    import torch
    times, units, labels = split
    T = T or T_BINS
    idx = np.random.permutation(len(labels)) if shuffle else np.arange(len(labels))
    for b0 in range(0, len(idx), batch_size):
        sel = idx[b0:b0+batch_size]
        x = torch.zeros(len(sel), T, N_IN)
        for i, j in enumerate(sel):
            tt = times[j]; uu = units[j]
            keep = tt < MAX_TIME
            tb = np.clip((tt[keep] / MAX_TIME * T).astype(int), 0, T-1)
            x[i, tb, uu[keep]] = 1.0
        yield x.to(device), torch.as_tensor(labels[sel]).to(device)

## Neuron cells
All cells share the triangle surrogate and expose `(spikes, spike_count)` over a `(B, T, N)` input current sequence plus a recurrent weight applied to their own output. Per-neuron hyperparameters follow each model's authors (TC-LIF: threshold 1.5, γ = 0.5, learnable decays initialized at 0; DH-LIF: 4 branches with learnable timing factors spread across timescales); the skeleton, optimizer, schedule, and loss are identical for all.

In [ ]:
import torch, torch.nn as nn, math

class Triangle(torch.autograd.Function):
    gamma = 1.0
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return (x >= 0).float()
    @staticmethod
    def backward(ctx, g):
        (x,) = ctx.saved_tensors
        return g * torch.clamp(1.0 - x.abs() / Triangle.gamma, min=0.0)
spike_fn = Triangle.apply

def decay(tau): return math.exp(-1.0 / tau)   # per-step decay, tau in steps

class LIFCell(nn.Module):
    th = 1.0
    def __init__(self, N): super().__init__(); self.N = N; self.am = decay(20)
    def init(self, B, dev): self.v = torch.zeros(B, self.N, device=dev)
    def forward(self, I):
        self.v = self.am * self.v + I
        s = spike_fn(self.v - self.th)
        self.v = self.v - s.detach() * self.th          # soft reset
        return s

class ALIFCell(nn.Module):
    th = 1.0; beta = 1.6
    def __init__(self, N): super().__init__(); self.N = N; self.am = decay(20); self.aa = decay(200)
    def init(self, B, dev):
        self.v = torch.zeros(B, self.N, device=dev); self.a = torch.zeros(B, self.N, device=dev)
    def forward(self, I):
        self.v = self.am * self.v + I
        th = self.th + self.beta * self.a
        s = spike_fn(self.v - th)
        self.v = self.v - s.detach() * th.detach()
        self.a = self.aa * self.a + s.detach()
        return s

class TCLIFCell(nn.Module):
    """Official TC-LIF dynamics (ZhangShimin1/TC-LIF): v1 = v1 - sig(d0)*v2 + I ; v2 = v2 + sig(d1)*v1;
       spike on v2 >= 1.5; soft reset v1 -= gamma*s, v2 -= th*s; d learnable, init 0."""
    th = 1.5; gamma_r = 0.5
    def __init__(self, N):
        super().__init__(); self.N = N
        self.d = nn.Parameter(torch.zeros(2))
    def init(self, B, dev):
        self.v1 = torch.zeros(B, self.N, device=dev); self.v2 = torch.zeros(B, self.N, device=dev)
    def forward(self, I):
        self.v1 = self.v1 - torch.sigmoid(self.d[0]) * self.v2 + I
        self.v2 = self.v2 + torch.sigmoid(self.d[1]) * self.v1
        s = spike_fn(self.v2 - self.th)
        self.v1 = self.v1 - s * self.gamma_r
        self.v2 = self.v2 - s * self.th
        return s

class DHLIFCell(nn.Module):
    """Re-implementation per Zheng et al. 2024: k dendritic branches with learnable timing factors;
       input current is split across branches; membrane integrates the summed branch currents."""
    th = 1.0; K = 4
    def __init__(self, N):
        super().__init__(); self.N = N; self.am = decay(20)
        init_a = torch.tensor([0.5, 0.8, 0.95, 0.99])
        logit = torch.log(init_a / (1 - init_a))
        self.branch_logit = nn.Parameter(logit.view(self.K, 1).repeat(1, N))   # (K, N)
        self.mix = nn.Parameter(torch.ones(self.K, N) / self.K)
    def init(self, B, dev):
        self.i = torch.zeros(B, self.K, self.N, device=dev); self.v = torch.zeros(B, self.N, device=dev)
    def forward(self, I):
        ad = torch.sigmoid(self.branch_logit)                                   # (K, N)
        self.i = ad.unsqueeze(0) * self.i + I.unsqueeze(1) / self.K
        self.v = self.am * self.v + (self.mix.unsqueeze(0) * self.i).sum(1)
        s = spike_fn(self.v - self.th)
        self.v = self.v - s.detach() * self.th
        return s

class PGLIFCell(nn.Module):
    """PG-LIF v2 (manuscript Eqs. 6-10, ML current convention). Feedforward current drives BOTH
       compartments: the soma directly and the dendrite for plateau initiation; the plateau adds the
       slow pathway kappa*p on top. Dendrite-only feedforward routing (the v1 defect) is an ablation.
       Learnable: alpha_p (per neuron), kappa."""
    th = 1.0; th_d = 1.0; P0 = 1.0; beta = 1.0
    def __init__(self, N, tref_p=10):
        super().__init__(); self.N = N
        self.am = decay(20); self.ad = decay(20); self.aa = decay(200)
        ap0 = decay(T_BINS / 2)
        self.ap_logit = nn.Parameter(torch.full((N,), math.log(ap0 / (1 - ap0))))
        self.kappa = nn.Parameter(torch.ones(N))
        self.tref_p = tref_p
    def init(self, B, dev):
        z = lambda: torch.zeros(B, self.N, device=dev)
        self.vs, self.vd, self.p, self.a = z(), z(), z(), z()
        self.rp = torch.zeros(B, self.N, device=dev)
    def forward(self, I_ff, I_rec):
        self.vd = self.ad * self.vd + I_ff
        ed = spike_fn(self.vd - self.th_d) * (self.rp == 0).float()
        self.rp = torch.clamp(self.rp - 1, min=0) + ed.detach() * self.tref_p
        self.p = torch.sigmoid(self.ap_logit) * self.p + self.P0 * ed
        self.vs = self.am * self.vs + I_ff + I_rec + self.kappa * self.p
        th = self.th + self.beta * self.a
        s = spike_fn(self.vs - th)
        self.vs = self.vs - s.detach() * th.detach()
        self.a = self.aa * self.a + s.detach()
        return s

## Network skeleton and training
One recurrent spiking layer (identical for all neurons), leaky readout, cross-entropy on the readout potential summed over time. Adam, StepLR (0.1x at the scheduled epochs), gradient clipping at 5. Spike counts are recorded at every evaluation.

In [ ]:
class RecSNN(nn.Module):
    def __init__(self, cell_name):
        super().__init__()
        self.cell_name = cell_name
        self.w_in = nn.Linear(N_IN, HIDDEN)
        self.w_rec = nn.Linear(HIDDEN, HIDDEN, bias=False)
        self.cell = {'LIF': LIFCell, 'ALIF': ALIFCell, 'TCLIF': TCLIFCell,
                     'DHLIF': DHLIFCell, 'PGLIF': PGLIFCell}[cell_name](HIDDEN)
        self.w_out = nn.Linear(HIDDEN, N_OUT)
        self.a_out = decay(20)
        nn.init.orthogonal_(self.w_rec.weight)
    def forward(self, x):                       # x: (B, T, N_IN)
        B, T, _ = x.shape; dev = x.device
        self.cell.init(B, dev)
        s = torch.zeros(B, HIDDEN, device=dev)
        out = torch.zeros(B, N_OUT, device=dev); vo = torch.zeros(B, N_OUT, device=dev)
        n_spk = 0.0
        for t in range(T):
            iff = self.w_in(x[:, t]); irec = self.w_rec(s)
            s = self.cell(iff, irec) if self.cell_name == 'PGLIF' else self.cell(iff + irec)
            n_spk = n_spk + s.detach().sum()
            vo = self.a_out * vo + self.w_out(s)
            out = out + vo
        return out, n_spk / B

def evaluate(model, device):
    model.eval(); correct = tot = 0; spk = 0.0; nb = 0
    with torch.no_grad():
        for x, y in batches(TE, 256, shuffle=False, device=device):
            out, ns = model(x)
            correct += (out.argmax(1) == y).sum().item(); tot += len(y)
            spk += ns.item(); nb += 1
    return correct / tot, spk / nb

def train_one(model_name, seed, device):
    res_file = os.path.join(OUT, f'{model_name}_s{seed}.json')
    if os.path.exists(res_file):
        print(f'[skip] {model_name} seed {seed} already done'); return json.load(open(res_file))
    torch.manual_seed(seed); np.random.seed(seed)
    model = RecSNN(model_name).to(device)
    n_par = sum(p.numel() for p in model.parameters())
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    sch = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=SCHEDULE, gamma=0.1)
    crit = nn.CrossEntropyLoss()
    best, best_spk, hist = 0.0, 0.0, []
    for ep in range(EPOCHS):
        model.train(); t0 = time.time()
        for x, y in batches(TR, BATCH, shuffle=True, device=device):
            opt.zero_grad()
            out, _ = model(x)
            loss = crit(out, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
        sch.step()
        acc, spk = evaluate(model, device)
        hist.append({'epoch': ep, 'test_acc': acc, 'spikes': spk})
        if acc > best:
            best, best_spk = acc, spk
            torch.save(model.state_dict(), os.path.join(OUT, f'{model_name}_s{seed}_best.pt'))
        print(f'{model_name} s{seed} ep{ep:03d}  acc {acc:.4f} (best {best:.4f})  '
              f'spk/sample {spk:.0f}  {time.time()-t0:.0f}s')
    res = {'model': model_name, 'seed': seed, 'best_test_acc': best,
           'spikes_per_sample': best_spk, 'params': n_par, 'history': hist}
    json.dump(res, open(res_file, 'w'), indent=2)
    return res

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cpu': print('WARNING: no GPU detected - this will be very slow. Switch runtime to GPU.')
results = []
for m in MODELS:
    for sd in SEEDS:
        results.append(train_one(m, sd, device))

## Aggregate results and decision gate

In [ ]:
import matplotlib.pyplot as plt
from collections import defaultdict
agg = defaultdict(list)
for r in results: agg[r['model']].append(r)
rows = []
for m in MODELS:
    accs = np.array([r['best_test_acc'] for r in agg[m]])
    spks = np.array([r['spikes_per_sample'] for r in agg[m]])
    rows.append({'model': m, 'acc_mean': accs.mean(), 'acc_std': accs.std(),
                 'spikes': spks.mean(), 'params': agg[m][0]['params'], 'n_seeds': len(accs)})
    print(f"{m:6s}  acc {accs.mean()*100:.2f} +- {accs.std()*100:.2f} %   "
          f"spikes/sample {spks.mean():.0f}   params {agg[m][0]['params']}")
json.dump(rows, open(os.path.join(OUT, 'aggregate.json'), 'w'), indent=2)

plt.figure(figsize=(7, 4))
plt.bar([r['model'] for r in rows], [r['acc_mean']*100 for r in rows],
        yerr=[r['acc_std']*100 for r in rows], capsize=4)
plt.ylabel('SHD test accuracy (%)'); plt.title(f'P1 ({RUN_TAG})')
plt.tight_layout(); plt.savefig(os.path.join(OUT, 'fig_P1_accuracy.png'), dpi=300); plt.show()

pg = next((r for r in rows if r['model'] == 'PGLIF'), None)
tc = next((r for r in rows if r['model'] == 'TCLIF'), None)
dh = next((r for r in rows if r['model'] == 'DHLIF'), None)
if pg and (tc or dh):
    ref = max([r for r in (tc, dh) if r], key=lambda r: r['acc_mean'])
    gap = (pg['acc_mean'] - ref['acc_mean']) * 100
    noise = 2 * max(pg['acc_std'], ref['acc_std']) * 100 if len(SEEDS) > 1 else 1.5
    verdict = ('PASSED - PG-LIF within noise of the strongest two-compartment baseline or better; proceed to P2'
               if gap >= -noise else
               'NOT MET - PG-LIF clearly below the strongest baseline; consult fallback in plan Sec. 7 before P2')
    print(f'\nDECISION GATE P1: gap to {ref["model"]} = {gap:+.2f} pp (noise band {noise:.2f} pp) -> {verdict}')
if not PAPER_MODE:
    print('\nNOTE: quick mode. Table 1 of the manuscript requires PAPER_MODE = True (T=250, 100 epochs, seeds 0-4).')

### Next steps
1. Quick pass: only PG-LIF retrains (baselines are reused from v1). Check that PG-LIF now trains normally and where it lands relative to TC-LIF/DH-LIF.
2. If the ranking is sensible, set `PAPER_MODE = True` for the Table 1 run (T = 250, 100 epochs, seeds 0-4; resumable).
3. **Phase P2 next:** ablation (a) — plateau replaced by a second ALIF adaptation variable — plus ablation of the v1 routing (dendrite-only feedforward), which this defect showed is genuinely destructive and therefore worth reporting.
4. Before manuscript use, validate the DH-LIF re-implementation against a published DH-LIF SHD number.